## **Problem 1: Lazy Evaluation with Generators**
A generator produces values one at a time instead of storing the entire result in memory.

### Task

1. Write a generator function `count_up_to(n)` that yields numbers from `1` to `n`.
2. Use a `for` loop to print the generated values.
3. Print the type of the object returned by the generator function.

### Expected behaviour

Example:
```python
count_up_to(5)
```

Output:

1
2
3
4
5

### Reflection Questions
- Why is a generator more memory efficient than returning a full list?
- When would you design a function as a generator rather than returning a collection?
- What is the effect of an infinite loop (`while True`) inside a generator function?
- What happens if a generator contains both `yield` and `return` statements?
---

In [137]:
def count_up_to(n):
    for i in range(1, n+1):
        yield i
        
for i in count_up_to(5):
    print(i)
    
    
#A generator produces values one at a time instead of storing the entire result in memory
#When dealing with large datasets, generators can be more memory-efficient than lists / collections
#As long as the condition is true, it will keep yielding values indefinitely as long as next() is called.
#If there is yield in the function, then it is a generator. Return stops the iterator, raising the StopIteration exception


1
2
3
4
5


## **Problem 2: Data Pipeline using Generators**

Generators are often composed to create efficient data pipelines.

Consider a list of text entries:

data = ["apple", "", "banana", "cherry", "", "date"]

### Task

Create a generator pipeline consisting of three steps:

1. `filter_empty()`  
   Remove empty strings.

2. `make_uppercase()`  
   Convert each word to uppercase.

3. `add_length()`  
   Return a tuple containing the word and its length.

The final output should look like:

```python
('APPLE', 5)
('BANANA', 6)
('CHERRY', 6)
('DATE', 4)
```
### Question
What advantage does this pipeline approach provide when working with very large datasets?

---


In [138]:
data = ["apple", "", "banana", "cherry", "", "date"]


def filter_empty(data):
    d = (item for item in data if item.strip())
    # data.clear()
    # data.append(d)
    for item in d:
        yield item
    
def make_uppercase(items):
    for i in filter_empty(items):
        yield i.upper()
    

def add_length(items):
    for i in make_uppercase(items):
        yield (i, len(i))
        
for i in filter_empty(data):
    print(i)
    
for i in make_uppercase(data):
    print(i)
    
for i in add_length(data):
    print(i)





apple
banana
cherry
date
APPLE
BANANA
CHERRY
DATE
('APPLE', 5)
('BANANA', 6)
('CHERRY', 6)
('DATE', 4)


## **Problem 3: Writing a Decorator**

Decorators modify the behaviour of a function without changing its source code.

### Task

1. Write a decorator called `log_calls`.
2. The decorator should print: Function <function_name> was called
3. Apply the decorator to a function `add(a, b)`.

### Expected behaviour

Calling:

add(3, 4)

Output:

Function add was called
7

### Question

Why are decorators useful in large software systems?

---


In [139]:
from functools import wraps


def log_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Function {func.__name__} was called")
        return func(*args, **kwargs)
    return wrapper


@log_calls
def add(a, b):
    return a + b

res = add(3, 5)

Function add was called


## **Problem 4: Counting Function Calls**
Decorators can also store state.

### Task

Create a decorator `count_calls` that keeps track of how many times a function is executed.

Each call should print:
- Call number X
- Apply the decorator to a function `greet(name)`.

### Example output

Call number 1
Hello Alice

Call number 2
Hello Bob

---

In [140]:
# count = 0
def count_calls(func):
    count = 0
    @wraps(func)
    def wrapper(*args, **kwargs):
        nonlocal count
        count += 1
        print(f"Count: {count}")
        return func(*args, **kwargs)
    return wrapper

@count_calls
def greet(name):
    print(f"Hello {name}")
    
greet("Fred")
greet("Fred")

Count: 1
Hello Fred
Count: 2
Hello Fred


## **Problem 5: Context Managers**

Context managers control resource acquisition and release.

### Task

Create a context manager called `managed_resource(name)` using `contextlib.contextmanager`.

The behaviour should be:

Opening <name>
Working with <name>
Closing <name>

### Example usage
```python
with managed_resource("database") as db:
    print("Working with", db)
```
### Question

Why are context managers useful for handling resources like files or database connections?

---

In [141]:
from contextlib import contextmanager


@contextmanager

def managed_resource(name):
    print("Opening")
    try:
        yield name
    finally:
        print(f"Closing {name}")
        
with managed_resource("database") as db:
    print(f"Working with {db}")   

Opening
Working with database
Closing database


## **Problem 6: Performance Monitoring**


In large applications, it is useful to measure how long functions take to execute.

### Task

Create a decorator called `timer` that:

1. Measures the execution time of a function
2. Prints:
```python 
Function <name> executed in <time> seconds
```
Apply it to a function that computes the sum of numbers from 1 to 1,000,000.

### Question
Why is a decorator better than manually inserting timing code inside every function?

---

In [142]:
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        func(*args, **kwargs)
        end = time.time()
        print(f"Function: {func.__name__} executed in {end - start}")
    return wrapper


@timer
def sum_range():
    sum(range(1_000_000))

exp = sum_range()

Function: sum_range executed in 0.006837606430053711


## **Problem 7: Refactoring Code**

Rewrite the following code using a more expressive Python style using `map` & `filter`.
```python
Original code:

result = []
for x in range(10):
    if x % 2 == 0:
        result.append(x * x)
```

### Task

Rewrite this logic using a more concise Python approach.

### Discussion

Why does expressive code improve maintainability?

---

In [143]:
result = list(map(lambda x: x * x, filter(lambda x: x % 2 == 0, range(10))))
print(list(result))

[0, 4, 16, 36, 64]


### Problem 1: Data Transformation Pipeline using `map`, `filter`, and `lambda`

Consider a system that processes a stream of sensor readings collected from multiple devices. 
Each reading is represented as a tuple `(device_id, value)`.

Some readings are invalid (negative values) and must be discarded. The remaining values must be normalised and transformed for further analysis.

#### Scenario

Given a list of readings:

```python
readings = [
    ("A", 10),
    ("B", -3),
    ("A", 25),
    ("C", 0),
    ("B", 15)
]
```

1. Use `filter` and a `lambda` function to remove invalid readings (values less than 0).
2. Use `map` and a `lambda` function to convert each valid reading into the form:
   ```python
   (device_id, value * 2)
   ```
3. Store the final result as a list.

#### Expected Transformation
```python
("A", 10) → ("A", 20)
```

#### Discussion
- What are the advantages of using `map` and `filter` instead of explicit loops in functional-style data pipelines?
- What are the trade-offs in terms of readability and debugging complexity?
---

#### Extension 1: Generator Expressions

Rewrite the transformation pipeline using a generator expression rather than `map` and `filter`.

Construct a generator that:
- filters out invalid readings (value < 0)
- yields `(device_id, value * 2)` lazily
---

#### Extension 2: Decorators

Define a decorator `log_execution(func)` that prints:

```python
Entering <function_name>
Exiting <function_name>
```

Apply it to a function that processes the readings pipeline.

---

#### Extension 3: Function Wrapping

Define a wrapper function `safe_process(readings)` that:
- validates input structure
- ensures all elements are tuples of length 2
- raises a controlled exception otherwise
- returns the processed result
---

In [144]:
readings = [
    ("A", 10),
    ("B", -3),
    ("A", 25),
    ("C", 0),
    ("B", 15)
]

filtered_readings = list(filter(lambda r: r[1] >= 0, readings))
print(filtered_readings)

transformed_readings = map(lambda tr: (tr[0], tr[1] * 2), filtered_readings)
print(list(transformed_readings))


[('A', 10), ('A', 25), ('C', 0), ('B', 15)]
[('A', 20), ('A', 50), ('C', 0), ('B', 30)]


In [145]:
def log_execution(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            print(f"Entering {func.__name__}")
            func(*args, **kwargs)
        finally:
            print(f"Exiting {func.__name__}")
    return wrapper
  
@log_execution          
def safe_process(readings):
    try:
        if type(readings) != list:
            raise ValueError("readings must be a list")
        if not all(type(r) == tuple and len(r) == 2 for r in readings):
            raise ValueError("readings must be a list of tuples of length 2")
        print(list(map(lambda r: (r[0], r[1] * 2), filter(lambda r: r[1] >= 0, readings))))
        return list(map(lambda r: (r[0], r[1] * 2), filter(lambda r: r[1] >= 0, readings)))
    except ValueError as e:
        print(f"Error: {e}")
        return

log_ex = safe_process(readings)
# log_ex(readings)

Entering safe_process
[('A', 20), ('A', 50), ('C', 0), ('B', 30)]
Exiting safe_process


### Problem 2: Aggregation using `reduce` and higher-order functions

Consider a simple financial system that records daily transactions. Each transaction is represented as a numeric value, where positive numbers indicate income and negative numbers indicate payments.

#### Task

Given the list:

```python
transactions = [100, -20, 50, -10, 200, -30]
```

1. Use `filter` with a `lambda` function to separate only positive transactions.
2. Use `map` with a `lambda` function to apply a processing fee of 5% to each positive transaction.
3. Use `reduce` from `functools` with a `lambda` function to compute the total sum of the processed transactions.

#### Expected output

```python
Positive transactions: [100, 50, 200]
After 5% fee: [95, 47.5, 190]
Final total: sum of all values
```

#### Discussion

- What role does `reduce` play in functional programming compared to iterative accumulation using loops?
- Under what conditions does `reduce` become less readable or less appropriate than explicit iteration?
---

#### Extension 1: Generator Expressions

Rewrite the transaction pipeline using a **generator expression** that:
- filters positive transactions
- applies a 5% processing fee lazily

Question: Explain the implications for memory efficiency.

---

#### Extension 2: Decorators

Create a decorator `timeit(func)` that measures execution time of the reduction pipeline and prints the elapsed time.

---

#### Extension 3: Function Wrapping

Encapsulate the full transaction workflow into a function:

```python
def process_transactions(transactions):
    ...
```
Ensure that the function:
- applies filtering, mapping, and reduction in a clean pipeline
- returns the final aggregated value
---

In [146]:
from functools import reduce


transactions = [100, -20, 50, -10, 200, -30]
positive_transactions = list(filter(lambda x: x > 0, transactions))
print(positive_transactions)

processed_transactions = list(map(lambda x: x - (x*(5/100)), positive_transactions))
print(processed_transactions)

sum_transactions = reduce(lambda x, y: x + y, processed_transactions)
print(sum_transactions)



[100, 50, 200]
[95.0, 47.5, 190.0]
332.5


In [147]:
transactions = [100, -20, 50, -10, 200, -30]


# def filter_transactions(transactions):
#     for t in transactions:
#         if t > 0:
#             yield t
   
   
            
# def processed_transaction(transaction):
#     return transaction - (transaction * (5/100))
    
# for t in filter_transactions(transactions):
#     processed = processed_transaction(t)
#     print(processed)

process_transaction_pipeline = (t - (t * 5/100) for t in transactions if t > 0)
print(list(process_transaction_pipeline))
    

    

    


[95.0, 47.5, 190.0]


In [148]:
def timeit(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        res = func(*args, **kwargs)
        end = time.time()
        print(f"{func.__name__} took {end - start:.6f}s")
        return res
    return wrapper
    

In [149]:
@timeit
def process_transactions(transactions):
    filter_transactions = list(filter(lambda t: t > 0, transactions))
    processed_filtered = list(map(lambda t: t - (t * 5/100), filter_transactions))
    sum_all = reduce(lambda x, y: x + y, processed_filtered)
    print(sum_all)

process_transactions(transactions)

332.5
process_transactions took 0.000137s
